# RECOL·LECCIÓ D'ENTITATS

Aquest notebook és el **Pas 0** del pipeline: produeix la llista d'entitats candidates.

Fonts:

- **A. Errors mesurats del model per defecte** — fallades reals observades.
- **B. NER automàtic sobre corpus** (spaCy) — *generador* de candidats. Avui no es fusiona a la llista final: sense un verificador que confirmi quins fallen de veritat, només dilueix.
- **C. Fragmentació del tokenizer de Whisper** — prior barat per **ordenar** els candidats de B.


In [2]:
import json
import re
from collections import defaultdict
from pathlib import Path

ROOT = Path("/media/ugiat/dd2/projects/nerea/sintetic_dataset/Sintetic-dataset")
candidatos = defaultdict(lambda: {"fuentes": set(), "freq": 0, "docs": set()})


def _es_hoja(o) -> bool:
    """Un registro 'hoja' es un dict sin dicts anidados: cabe en una línea."""
    return isinstance(o, dict) and all(
        not isinstance(v, dict) and not (isinstance(v, list) and any(isinstance(x, dict) for x in v))
        for v in o.values()
    )


def json_compacto(obj, nivel: int = 0, sangria: int = 2) -> str:
    """Como json.dumps(indent=2) pero dejando cada registro en una sola línea,
    para poder ojear y grepear los ficheros sin scrollear cientos de líneas."""
    pad, pad2 = " " * (nivel * sangria), " " * ((nivel + 1) * sangria)
    if isinstance(obj, dict):
        if not obj:
            return "{}"
        if _es_hoja(obj):
            return json.dumps(obj, ensure_ascii=False)
        cuerpo = ",\n".join(
            f"{pad2}{json.dumps(k, ensure_ascii=False)}: {json_compacto(v, nivel + 1, sangria)}"
            for k, v in obj.items()
        )
        return "{\n" + cuerpo + f"\n{pad}}}"
    if isinstance(obj, list):
        if not obj:
            return "[]"
        if all(not isinstance(x, (dict, list)) for x in obj):
            return json.dumps(obj, ensure_ascii=False)
        cuerpo = ",\n".join(f"{pad2}{json_compacto(x, nivel + 1, sangria)}" for x in obj)
        return "[\n" + cuerpo + f"\n{pad}]"
    return json.dumps(obj, ensure_ascii=False)


## Font A — Detectar errors del model per defecte

Parteix de `lab/entitats/entitats_fallades/entidades_erroneas.json`, que produeix `lab/entitats/entitats_fallades/evaluate.py` alineant les transcripcions de `large-v3` amb el ground truth d'àudios de RNE. Barreja tres problemes diferents i només un es corregeix amb dades sintètiques.

| Tipus de fallada | Exemple | L'arregla el dataset sintètic? |
|---|---|---|
| Confusió lèxica/fonètica | `llorca -> yorca` | **Sí**, és l'objectiu |
| Omissió de segment sencer | `radio` / `nacional` / `españa` a la careta cantada | No, és robustesa a música/VAD |
| Normalització | `diez -> 10` | No, s'arregla en post-procés |

Els blocs següents apliquen un embut de descartes automàtics i reconstrueixen els noms compostos.

> **Pendent de validació humana/LLM:** en bastants casos el ground truth és el que està mal escrit i el model té raó.


In [3]:
# A.1 — Cargar los errores detectados y calcular la señal de "nombre común"
import sys
from collections import Counter

EVAL_DIR = ROOT / "lab/entitats/entitats_fallades"  # aquí vive entidades_erroneas.json
sys.path.insert(0, str(ROOT / "src"))                 # evaluate.py vive aquí ahora
from evaluate import PAIRS, OMITIDA, load_raw, norm, strip_accents, similitud, similitud_del_cambio

errores = json.loads((EVAL_DIR / "entidades_erroneas.json").read_text(encoding="utf-8"))

# si una palabra aparece TAMBIÉN en
# minúscula dentro del ground truth, es un nombre común capitalizado por posición
# (Radio, Día, Horas, Secretario) y no una entidad. Evita mantener una stoplist a mano.
uso_mayus, uso_minus = Counter(), Counter()
for _nombre, _stem, gt_path in PAIRS:
    print(f"Procesando {gt_path} ...")
    for frase in re.split(r"(?<=[.!?])\s+", load_raw(gt_path)):
        for i, palabra in enumerate(frase.split()):
            tok = palabra.strip(',.;:¿?¡!()"\'«»…—-')
            if not tok or i == 0:  # la primera palabra de la frase siempre va capitalizada
                continue
            n = norm(tok)
            if n:
                (uso_mayus if tok[0].isupper() else uso_minus)[n] += 1

total_errores = sum(sum(n for _, n in e["mal_transcrita"]) for e in errores)
print(f"{len(errores)} entidades con error / {total_errores} errores totales")
print(f"vocabulario del ground truth: {len(uso_mayus)} formas capitalizadas, {len(uso_minus)} en minúscula")

Procesando /media/ugiat/dd2/projects/nerea/sintetic_dataset/Sintetic-dataset/lab/inputs/RNE/groundtruths/ground_truth_R1_24HORAS.json ...
Procesando /media/ugiat/dd2/projects/nerea/sintetic_dataset/Sintetic-dataset/lab/inputs/RNE/groundtruths/ground_truthMEDIODIA.txt ...
Procesando /media/ugiat/dd2/projects/nerea/sintetic_dataset/Sintetic-dataset/lab/inputs/RNE/groundtruths/ground_truthNEUDC.json ...
Procesando /media/ugiat/dd2/projects/nerea/sintetic_dataset/Sintetic-dataset/lab/inputs/RNE/groundtruths/ground_truth_R1_EL-ULTIMO-TREN.json ...
338 entidades con error / 609 errores totales
vocabulario del ground truth: 1578 formas capitalizadas, 8435 en minúscula


In [4]:
# A.2 — Embudo de descartes automáticos
from num2words import num2words

SIM_MIN = 0.65   # parecido mínimo entidad <-> transcripción errónea (similitud() de evaluate.py:
                  # grafía + fonética, ignorando cómo Whisper separe las palabras)
LONG_MIN = 4     # las palabras muy cortas dan demasiados falsos positivos al alinear


def n_tildes(palabra: str) -> int:
    return sum(1 for a, b in zip(palabra, strip_accents(palabra)) if a != b)


def es_forma_numerica_de(correcta: str, v: str) -> bool:
    """True si v es 'correcta' escrito en cifras (diez -> 10), no solo un digito suelto
    que coincidio por ruido de alineamiento (club -> 1)."""
    if not v.isdigit():
        return False
    forma_escrita = strip_accents(num2words(int(v), lang="es"))
    return strip_accents(correcta) == forma_escrita


def motivo_descarte(correcta: str, variantes: list):
    """Devuelve el motivo por el que la entidad NO sirve, o None si es candidata válida."""
    subs = [(v, n) for v, n in variantes if v != OMITIDA]

    if any(es_forma_numerica_de(correcta, v) for v, _ in variantes):
        return "cifra"  # diez->10: diferencia de normalización, no error léxico
    if uso_minus[correcta] > 0 and uso_minus[correcta] >= uso_mayus[correcta] * 0.5:
        return "nombre_comun"  # aparece en minúscula en el GT
    if any(strip_accents(v) == strip_accents(correcta) and n_tildes(v) > n_tildes(correcta) for v, _ in subs):
        return "gt_sin_tilde"  # el modelo acentúa y el GT no: el GT es el que está mal
    if not subs:
        return "solo_omitida"  # el modelo se saltó el segmento entero (careta/música)
    if max(similitud(correcta, v) for v, _ in subs) < SIM_MIN:
        return "ruido_alineamiento"  # sustitución sin ningún parecido fonético
    if len(correcta) < LONG_MIN:
        return "muy_corta"
    return None


fuente_a, descartes = [], defaultdict(list)
for e in errores:
    correcta, variantes = e["bien_escrita"], e["mal_transcrita"]
    registro = {"entidad": correcta, "veces": sum(n for _, n in variantes), "variantes": variantes}
    motivo = motivo_descarte(correcta, variantes)
    if motivo:
        descartes[motivo].append(registro)
    else:
        subs = [v for v, _ in variantes if v != OMITIDA]
        registro["sim_max"] = round(max(similitud(correcta, v) for v in subs), 3)
        fuente_a.append(registro)

print(f"{'motivo':22} {'ent':>4} {'err':>4}   ejemplos")
print("-" * 78)
for motivo, items in sorted(descartes.items(), key=lambda kv: -len(kv[1])):
    ejemplos = ", ".join(d["entidad"] for d in items[:5])
    print(f"{motivo:22} {len(items):4} {sum(d['veces'] for d in items):4}   {ejemplos}")
print("-" * 78)
print(f"{'CONSERVADAS':22} {len(fuente_a):4} {sum(d['veces'] for d in fuente_a):4}")

motivo                  ent  err   ejemplos
------------------------------------------------------------------------------
nombre_comun             33   56   dana, comunitat, radio, vida, secretario
ruido_alineamiento       18   22   miriam, earth, keane, love, times
solo_omitida             17   19   españa, isabel, consell, the, floor
gt_sin_tilde             12   24   paris, iñigo, genova, valles, alaves
muy_corta                11   15   vox, pol, pot, eña, pau
cifra                     4   15   diez, veinticuatro, tres, cinco
------------------------------------------------------------------------------
CONSERVADAS             243  458


In [5]:
# A.3 — Reconstruir entidades multipalabra
# Alinear token a token parte los nombres compuestos y deja fragmentos sueltos en la lista
# ('torre', 'corts', 'alert'). Buscamos en el GT secuencias de palabras capitalizadas
# seguidas para recuperar la forma completa ('Torre Blanca', 'Corts Valencianes').
MAX_TOKENS = 3               # más de 3 casi siempre es una racha que junta dos nombres
CORTE = set(",;:.()«»…!?")   # si el token acaba aquí, el nombre termina ahí ("Catarroja, Valencia")

formas_completas = defaultdict(Counter)
for _nombre, _stem, gt_path in PAIRS:
    for frase in re.split(r"(?<=[.!?])\s+", load_raw(gt_path)):
        crudos = frase.split()
        tokens = [t.strip(',.;:¿?¡!()"\'«»…—-') for t in crudos]
        corta_despues = [bool(set(t[-1:]) & CORTE) for t in crudos]

        i = 1  # saltamos la primera palabra de la frase
        while i < len(tokens):
            if not tokens[i][:1].isupper():
                i += 1
                continue
            j = i
            while j < len(tokens) and tokens[j][:1].isupper() and (j - i) < MAX_TOKENS:
                j += 1
                if corta_despues[j - 1]:
                    break
            if j - i >= 2:
                grupo = " ".join(tokens[i:j])
                for tok in tokens[i:j]:
                    if norm(tok):
                        formas_completas[norm(tok)][grupo] += 1
            i = max(j, i + 1)

for item in fuente_a:
    formas = formas_completas.get(item["entidad"])
    item["forma_completa"] = formas.most_common(1)[0][0] if formas else None

con_forma = [d for d in fuente_a if d["forma_completa"]]
print(f"{len(con_forma)}/{len(fuente_a)} entidades forman parte de un nombre compuesto\n")
for d in sorted(con_forma, key=lambda x: -x["veces"])[:20]:
    print(f"  {d['entidad']:16} -> {d['forma_completa']}")

print("\n--- Top entidades conservadas (sueltas o compuestas) ---")
for d in sorted(fuente_a, key=lambda x: -x["veces"])[:25]:
    variantes = ", ".join(v for v, _ in d["variantes"][:4])
    print(f"  {(d['forma_completa'] or d['entidad']):26} x{d['veces']:<3} sim={d['sim_max']:.2f}  [{variantes}]")

150/243 entidades forman parte de un nombre compuesto

  lamine           -> Lamine Yamal
  llorca           -> Pérez Llorca
  feijóo           -> Núñez Feijóo
  corts            -> Les Corts
  yamal            -> Lamine Yamal
  mazón            -> Carlos Mazón
  torre            -> Torre Blanca
  blanca           -> Torre Blanca
  radiogaceta      -> Radio Nacional Radiogaceta
  ibárruri         -> Dolores Ibárruri
  alert            -> Earth Alert
  juanfran         -> Juanfran Pérez Llorca
  básquet          -> Valencia Básquet
  gema             -> Gema Alfaro
  dolset           -> Pérez Dolset
  marc             -> Marc Márquez
  morant           -> Diana Morant
  joan             -> Joan Baldoví
  díaz             -> Díaz Ayuso
  japoel           -> Japoel Tel Aviv

--- Top entidades conservadas (sueltas o compuestas) ---
  Lamine Yamal               x26  sim=0.97  [lamín, la minyamal, alhamid, la miña]
  Pérez Llorca               x22  sim=0.87  [yorca, de yorca, york, yerka]
  

In [6]:
# A.4 — Volcar a `candidatos` y exportar el fichero auditable
# Varios fragmentos pueden colapsar en la misma entidad ('lamine' + 'yamal' -> 'Lamine Yamal'),
# así que las señales se acumulan en lugar de sobrescribirse.
for d in fuente_a:
    entidad = d["forma_completa"] or d["entidad"]
    reg = candidatos[entidad]
    reg["fuentes"].add("A_errores_modelo")
    reg["freq"] += d["veces"]
    reg["veces_error"] = reg.get("veces_error", 0) + d["veces"]
    reg["variantes_erroneas"] = sorted(
        set(reg.get("variantes_erroneas", [])) | {v for v, _ in d["variantes"] if v != OMITIDA}
    )
    reg["sim_max"] = max(reg.get("sim_max", 0), d["sim_max"])

# Guardamos también los descartes (no se borran) para poder auditar el embudo y
# reajustar umbrales sin tener que volver a correr evaluate.py.
salida_a = {
    "conservadas": sorted(fuente_a, key=lambda d: -d["veces"]),
    "descartadas": {
        motivo: sorted(items, key=lambda d: -d["veces"]) for motivo, items in descartes.items()
    },
}
path_a = ROOT / "lab/entitats/entidades_fuente_a.json"
path_a.write_text(json_compacto(salida_a), encoding="utf-8")

n_a = sum(1 for d in candidatos.values() if "A_errores_modelo" in d["fuentes"])
print(f"Fuente A: {len(fuente_a)} fragmentos -> {n_a} entidades únicas tras agrupar")
print(f"Exportado a {path_a}")
print(
    "\nPENDIENTE antes de generar audio: validar la grafía con revisión humana/LLM.\n"
    "El ground truth también tiene erratas y en esos casos el modelo acierta\n"
    "(mouseti/Musetti, vasconia/Baskonia, letour/Letur) — entrenarlas degradaría el modelo."
)

Fuente A: 243 fragmentos -> 227 entidades únicas tras agrupar
Exportado a /media/ugiat/dd2/projects/nerea/sintetic_dataset/Sintetic-dataset/lab/entitats/entidades_fuente_a.json

PENDIENTE antes de generar audio: validar la grafía con revisión humana/LLM.
El ground truth también tiene erratas y en esos casos el modelo acierta
(mouseti/Musetti, vasconia/Baskonia, letour/Letur) — entrenarlas degradaría el modelo.


### A.5 — Validació de grafia amb LLM

L'embut automàtic no pot decidir si la grafia correcta és `vasconia` o `Baskonia`: això és coneixement del món. Un LLM sí que pot, i de pas tipifica l'entitat (PERSONA / LLOC / ORG…) per poder-la expandir després per famílies.

Mateix patró que `dictionary.ipynb`: OpenAI amb Structured Outputs, `temperature=0`, `seed` fixa, per lots i amb control de cost.

**Dues coses que fan que això funcioni:**

1. **Context del ground truth.** A cada entitat se li adjunta la frase on apareix.
2. **Permetre dir "no ho sé".** El risc real és que el LLM inventi una grafia plausible per a un poble petit o un periodista local. El prompt l'obliga a retornar l'entrada tal qual amb `entidad_conocida: false` quan no reconeix l'entitat.


In [7]:
# A.5 — Cliente, esquema y prompt de validación
import configparser
import os
import time

from openai import APIConnectionError, APITimeoutError, InternalServerError, OpenAI, RateLimitError

MODEL_VALIDACION = "gpt-4o-mini"  # gpt-4o afina más en entidades poco conocidas
PRICE = {
    "gpt-4o-mini": {"in": 0.150, "cached_in": 0.075, "out": 0.600},
    "gpt-4o": {"in": 2.500, "cached_in": 1.250, "out": 10.000},
}

key = os.environ.get("OPENAI_API_KEY")
cfg = configparser.ConfigParser()
cfg.read(ROOT / "others/config.ini")
if "OPENAI" in cfg and "KEY" in cfg["OPENAI"]:
    key = cfg["OPENAI"]["KEY"]
if not key:
    raise RuntimeError("No se ha encontrado OPENAI_API_KEY ni en env ni en others/config.ini")
client = OpenAI(api_key=key, base_url=os.environ.get("OPENAI_BASE_URL"))

# Contexto: la frase del ground truth donde aparece la entidad. Sin esto el modelo no
# puede desambiguar (Vasconia región vs Baskonia equipo de baloncesto).
frases_gt = []
for _nombre, _stem, gt_path in PAIRS:
    frases_gt += [f.strip() for f in re.split(r"(?<=[.!?])\s+", load_raw(gt_path)) if f.strip()]

# Las versiones "planas" (sin tildes, en minúscula) se derivan SIEMPRE de la lista
# de frases que se está buscando y se cachean por lista. Antes eran un parámetro
# aparte con default `frases_gt_planas`, y la Fuente B llamaba `buscar_contexto(ent,
# frases_b)` sin pasarlo: se buscaba sobre las 4750 frases del ground truth de A y
# se indexaba sobre las 166 del corpus de B -> IndexError, o contexto de otro corpus.
_cache_planas: dict[int, tuple[list, list]] = {}


def _planas(frases: list[str]) -> list[str]:
    clave = id(frases)
    if clave not in _cache_planas:
        # Se guarda también `frases` para mantener viva la referencia: si la lista se
        # liberase, Python podría reutilizar su id() y devolver un caché ajeno.
        _cache_planas[clave] = (frases, [strip_accents(f.lower()) for f in frases])
    return _cache_planas[clave][1]


def buscar_contexto(entidad: str, frases: list[str] | None = None, ventana: int = 220):
    frases = frases_gt if frases is None else frases
    frases_planas = _planas(frases)
    
    tokens = norm(entidad).split()
    claves = [strip_accents(norm(entidad))] + sorted((strip_accents(t) for t in tokens), key=len, reverse=True)
    
    for clave in claves:
        patron = re.compile(rf"\b{re.escape(clave)}\b")
        for i, plano in enumerate(frases_planas):
            m = patron.search(plano)
            if m:
                f = frases[i]
                ini = max(0, m.start() - ventana // 2)
                fin = min(len(f), ini + ventana)
                return ("…" if ini > 0 else "") + f[ini:fin].strip() + ("…" if fin < len(f) else "")
    return ""


SYSTEM_PROMPT_VALIDACION = """Eres un experto en entidades nombradas del ámbito informativo español (RNE).

Recibes entidades extraídas automáticamente de una transcripción de referencia (ground truth) hecha por humanos, junto con cómo las transcribió mal el modelo Whisper y la frase de contexto.

IMPORTANTE: el ground truth TAMBIÉN contiene erratas. En algunos casos la grafía correcta es la que produjo Whisper, no la del ground truth (ej.: ground truth "mouseti" pero el tenista se escribe "Musetti"; "vasconia" pero el equipo es "Baskonia"; "letour" pero el pueblo de Albacete es "Letur").

Para cada entrada devuelve:
- grafia_correcta: la grafía canónica y correctamente acentuada. Usa el contexto para desambiguar.
- gt_erroneo: true si la grafía canónica difiere de la entrada recibida (es decir, el ground truth estaba mal).
- tipo: PERSONA, LUGAR, ORGANIZACION, EVENTO, OTRO, o NO_ENTIDAD si no es un nombre propio.
- entidad_conocida: true si reconoces la entidad y confías en la `grafia_correcta` que devuelves; false si no la reconoces con seguridad (en ese caso sigue la REGLA CRÍTICA de abajo).
- motivo: una frase breve justificando la decisión.

REGLA CRÍTICA: si no reconoces la entidad con seguridad (topónimos menores, periodistas locales, nombres poco conocidos), NO inventes una grafía. Devuelve grafia_correcta igual a la entrada recibida, gt_erroneo=false y entidad_conocida=false. Es preferible mandarla a revisión humana que corromper el dataset."""

SCHEMA_VALIDACION = {
    "type": "object",
    "properties": {
        "entidades": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "entrada": {"type": "string"},
                    "grafia_correcta": {"type": "string"},
                    "gt_erroneo": {"type": "boolean"},
                    "tipo": {
                        "type": "string",
                        "enum": ["PERSONA", "LUGAR", "ORGANIZACION", "EVENTO", "OTRO", "NO_ENTIDAD"],
                    },
                    "entidad_conocida": {"type": "boolean"},
                    "motivo": {"type": "string"},
                },
                "required": [
                    "entrada", "grafia_correcta", "gt_erroneo",
                    "tipo", "entidad_conocida", "motivo",
                ],
                "additionalProperties": False,
            },
        }
    },
    "required": ["entidades"],
    "additionalProperties": False,
}

# Payload: una entrada por entidad conservada en la Fuente A
a_validar = [
    {
        "entrada": ent,
        "variantes_whisper": datos["variantes_erroneas"][:8],
        "veces": datos["veces_error"],
        "contexto": buscar_contexto(ent),
    }
    for ent, datos in candidatos.items()
    if "A_errores_modelo" in datos["fuentes"]
]
print(f"{len(a_validar)} entidades a validar con {MODEL_VALIDACION}")
print(json.dumps(a_validar[:3], indent=2, ensure_ascii=False))

227 entidades a validar con gpt-4o-mini
[
  {
    "entrada": "Lamine Yamal",
    "variantes_whisper": [
      "alhamid",
      "alhamid jamal",
      "cono sin la miña mal",
      "jamal",
      "la",
      "la mil",
      "la mim",
      "la minya"
    ],
    "veces": 41,
    "contexto": "…una hora, Radiogaceta, hoy entrevista con Luis de la Fuente, que va a hablar de toda la polémica a cuenta de Lamine Yamal."
  },
  {
    "entrada": "Pérez Llorca",
    "variantes_whisper": [
      "de yorca",
      "peret",
      "pérez llota mazón de verdad",
      "yerka",
      "yorca",
      "york"
    ],
    "veces": 24,
    "contexto": "…rante a sustituir a Mazón como presidente, su mano derecha estos años el alcalde de Finestrat, Juan Francisco Pérez Llorca."
  },
  {
    "entrada": "Núñez Feijóo",
    "variantes_whisper": [
      "contrafijo",
      "feijo",
      "feijo dijo",
      "feijó",
      "fejo",
      "fejó",
      "fijo",
      "fijón"
    ],
    "veces": 22,
    "contexto": "El 

In [8]:
# A.6 — validación por lotes
def _llamar_lote(lote, model=MODEL_VALIDACION, system_prompt=SYSTEM_PROMPT_VALIDACION, reintentos=4):
    for intento in range(reintentos):
        try:
            r = client.chat.completions.create(
                model=model,
                temperature=0,
                seed=42,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"Entidades a validar:\n{json.dumps(lote, ensure_ascii=False)}"},
                ],
                response_format={
                    "type": "json_schema",
                    "json_schema": {"name": "validacion_entidades", "strict": True, "schema": SCHEMA_VALIDACION},
                },
            )
            msg = r.choices[0].message
            if getattr(msg, "refusal", None):
                raise RuntimeError(f"Model refusal: {msg.refusal}")
            return json.loads(msg.content)["entidades"], r.usage
        except (RateLimitError, APIConnectionError, APITimeoutError, InternalServerError) as e:
            if intento == reintentos - 1:
                raise
            time.sleep(2**intento)
            print(f"  reintento {intento + 1} tras {type(e).__name__}")


def validar_grafias(items, mida_lot=25, model=MODEL_VALIDACION, system_prompt=SYSTEM_PROMPT_VALIDACION):
    resultados, coste, t0 = [], 0.0, time.time()
    for i in range(0, len(items), mida_lot):
        lote = items[i : i + mida_lot]
        salida, usage = _llamar_lote(lote, model, system_prompt)

        # El modelo no debe inventar ni omitir entradas (mismo control que dictionary.ipynb)
        esperadas = {d["entrada"] for d in lote}
        devueltas = {d["entrada"] for d in salida}
        if esperadas != devueltas:
            print(f"  [!] lote {i}: faltan {esperadas - devueltas} | sobran {devueltas - esperadas}")

        cached = getattr(getattr(usage, "prompt_tokens_details", None), "cached_tokens", 0) or 0
        p = PRICE[model]
        coste += (
            max(0, usage.prompt_tokens - cached) * p["in"] / 1_000_000
            + cached * p["cached_in"] / 1_000_000
            + usage.completion_tokens * p["out"] / 1_000_000
        )
        resultados += salida
        print(f"  lote {i // mida_lot + 1}: {len(salida)} entidades  (coste acumulado ${coste:.4f})")

    print(f"\nTotal: {len(resultados)} entidades en {time.time() - t0:.1f}s  |  coste ${coste:.4f}")
    return resultados

In [9]:
# ejecutar la validacion (esta celda GASTA API)
validaciones = validar_grafias(a_validar)

  lote 1: 25 entidades  (coste acumulado $0.0012)
  [!] lote 25: faltan {'Laszlo Krasnáhorkai'} | sobran set()
  lote 2: 24 entidades  (coste acumulado $0.0023)
  lote 3: 25 entidades  (coste acumulado $0.0034)
  lote 4: 25 entidades  (coste acumulado $0.0044)
  lote 5: 25 entidades  (coste acumulado $0.0055)
  lote 6: 25 entidades  (coste acumulado $0.0066)
  lote 7: 25 entidades  (coste acumulado $0.0077)
  lote 8: 25 entidades  (coste acumulado $0.0088)
  lote 9: 25 entidades  (coste acumulado $0.0099)
  lote 10: 2 entidades  (coste acumulado $0.0100)

Total: 226 entidades en 115.2s  |  coste $0.0100


In [10]:
def tipo_cambio(entrada: str, correcta: str) -> str:
    """El flag gt_erroneo del LLM no basta: nuestras entidades vienen en minúscula
    normalizada, así que una simple capitalización lo dispara. Clasificamos el cambio real."""
    a, b = entrada.strip().lower(), correcta.strip().lower()
    if a == b:
        return "solo_mayusculas"
    if strip_accents(a) == strip_accents(b):
        return "acento"
    if strip_accents(a) in strip_accents(b) or strip_accents(b) in strip_accents(a):
        return "expansion"
    return "grafia_distinta"

In [11]:
# A.7 — Aplicar la validación y separar lo que aún necesita ojo humano
por_entrada = {v["entrada"]: v for v in validaciones}

for v in validaciones:
    v["tipo_cambio"] = tipo_cambio(v["entrada"], v["grafia_correcta"])

for ent, datos in candidatos.items():
    v = por_entrada.get(ent)
    if not v or "A_errores_modelo" not in datos["fuentes"]:
        continue
    datos.update({
        "grafia_llm": v["grafia_correcta"],
        "tipo": v["tipo"],
        "tipo_cambio": v["tipo_cambio"],
        "entidad_conocida_llm": v["entidad_conocida"],
        "motivo_llm": v["motivo"],
    })

# NO se filtra por tipo_cambio. Se probó descartar las 'solo_mayusculas' (con el
# argumento de que Whisper es acústico y la capitalización no le aporta) y se midió:
# se cargaba 136 de 178 entidades, entre ellas 'Lamine Yamal', 'Carlos Mazón' y
# 'Núñez Feijóo'. El motivo es que `evaluate.py` normaliza a minúsculas, así que
# CUALQUIER entidad que el LLM solo tenga que capitalizar cae en ese grupo.
# `tipo_cambio` describe lo inconsistente que era el ground truth, no si el modelo
# falla: 'Lamine Yamal' tiene 41 errores acústicos ('minyamal', 'meñamal', 'llamal')
# y aun así su tipo_cambio es 'solo_mayusculas'.
# Guardarraíl anti-alucinación. El LLM reconoce (entidad_conocida=true) y "corrige"
# sustituyendo la entidad por OTRA distinta: medido en esta misma celda,
# 'Japoel Tel Aviv' -> 'Maccabi Tel Aviv' (dos equipos israelíes diferentes; el GT
# decía Hapoel). Entrenar eso le enseñaría a Whisper a decir el nombre equivocado.
# Una corrección de grafía legítima suena igual que el original (mouseti/Musetti,
# vasconia/Baskonia, letour/Letur >= 0.92); una sustitución por otra entidad no.
# No se descarta: se manda a revisión humana, que es donde se decide.
SIM_CAMBIO_MIN = 0.65

for v in validaciones:
    v["sim_cambio"] = round(similitud_del_cambio(v["entrada"], v["grafia_correcta"]), 3)

sospechosas = [v for v in validaciones
               if v["tipo"] != "NO_ENTIDAD" and v["sim_cambio"] < SIM_CAMBIO_MIN]

validadas = [v for v in validaciones
             if v["tipo"] != "NO_ENTIDAD" and v["entidad_conocida"]
             and v["sim_cambio"] >= SIM_CAMBIO_MIN]
revisar = [v for v in validaciones
           if v["tipo"] != "NO_ENTIDAD"
           and (not v["entidad_conocida"] or v["sim_cambio"] < SIM_CAMBIO_MIN)]
rechazadas = [v for v in validaciones if v["tipo"] == "NO_ENTIDAD"]
correcciones = [v for v in validadas if v["tipo_cambio"] != "solo_mayusculas"]

print(f"listas para el dataset : {len(validadas)}")
print(f"revisión humana        : {len(revisar)}")
print(f"rechazadas por el LLM  : {len(rechazadas)}")
print(f"\ncambios sobre la grafía del ground truth:")
for tc, n in Counter(v["tipo_cambio"] for v in validadas).most_common():
    print(f"  {tc:18} {n}")

print("\n--- Correcciones reales (el GT estaba mal, no es solo capitalizar) ---")
for v in sorted(correcciones, key=lambda x: x["tipo_cambio"]):
    print(f"  [{v['tipo_cambio']:15}] {v['entrada']:26} -> {v['grafia_correcta']}")

print(f"\n--- Sustituciones sospechosas (sim_cambio < {SIM_CAMBIO_MIN}): el LLM cambió la entidad, no la grafía ---")
for v in sorted(sospechosas, key=lambda x: x["sim_cambio"]):
    print(f"  [{v['sim_cambio']:.2f}] {v['entrada']:26} -> {v['grafia_correcta']:28} {v['motivo'][:45]}")

print("\n--- Cola de revisión humana ---")
for v in revisar:
    print(f"  {v['entrada']:26} [{v['tipo']}] {v['motivo'][:65]}")

print("\n--- Rechazadas por el LLM ---")
for v in rechazadas:
    print(f"  {v['entrada']:26} [{v['tipo']}] {v['motivo'][:65]}")

print("\n--- Reparto por tipo ---")
for tipo, n in Counter(v["tipo"] for v in validadas).most_common():
    print(f"  {tipo:15} {n}")

path_val = ROOT / "lab/entitats/entidades_fuente_a_validadas.json"
path_val.write_text(
    json_compacto({"validadas": validadas, "revisar_humano": revisar, "rechazadas": rechazadas}),
    encoding="utf-8",
)
print(f"\nGuardado en {path_val}")


listas para el dataset : 135
revisión humana        : 87
rechazadas por el LLM  : 4

cambios sobre la grafía del ground truth:
  solo_mayusculas    97
  grafia_distinta    28
  expansion          7
  acento             3

--- Correcciones reales (el GT estaba mal, no es solo capitalizar) ---
  [acento         ] baldovi                    -> Baldoví
  [acento         ] letúr                      -> Letur
  [acento         ] ortíz                      -> Ortiz
  [expansion      ] Díaz Ayuso                 -> Isabel Díaz Ayuso
  [expansion      ] Confederación Hidrográfica -> Confederación Hidrográfica del Júcar
  [expansion      ] Mohamed Shia               -> Mohamed Shia al-Sudani
  [expansion      ] País Valencián             -> País Valenciano
  [expansion      ] Imre Kertés                -> Imre Kertész
  [expansion      ] bbrañosera                 -> Brañosera
  [expansion      ] Carl Philippe              -> Carl Philipp
  [grafia_distinta] Valencia Básquet           -> Valenci

## Font B — NER automàtic sobre corpus (RTVE / Wikipedia)

Reutilitza el mateix model que `lab/proves_inicials/filter_entities.py` (`es_core_news_sm`) sobre el text cru de `lab/proves_inicials/proves/raw_text`, per generar candidats **abans** de passar per TTS/inferència.

**Estat: generador, no selector.** Executada sobre el corpus actual (3 documents) produeix ~230 entitats de les quals **només 1 coincideix** amb les que el model falla de veritat. És esperable: el NER troba entitats, no errors, i la majoria (`Gobierno`, `Estado`, `Constitución`) Whisper les transcriu perfectament.

Per això s'aboca a `candidatos_b`, **separat de `candidatos`**, i no entra a la llista final. El seu moment arriba quan hi hagi el verificador round-trip (vegeu propostes al final) i un corpus gran de veritat.

Filtres aplicats per reduir el soroll de `es_core_news_sm`: fora l'etiqueta `MISC` (la més sorollosa: retornava fragments de frase com `Su regulación` o `El eventual cese del presidente`), fora els spans que comencen per determinant o superen els 4 tokens, i fora les formes d'una sola paraula que apareixen en minúscula al propi corpus.

**Filtre reforçat.** Al soroll lèxic s'hi sumen falsos positius gramaticals que el filtre anterior no veia: spans que travessen un salt de frase/paràgraf arrossegant la primera paraula del bloc següent (`Ucrania\nEfectivamente`), sintagmes comuns sense cap paraula capitalitzada (`nueva generación`), i verbs/pronoms/adverbis colats al span encara que vagin en majúscula per anar al principi de frase (`Existe`, `Detrás`, `qué deberían`, `Athletic Club superó`). `motivo_descarte_b` els classifica igual que `motivo_descarte` a A.2.

**B.4-B.7 afegeixen una validació amb LLM** anàloga a la d'A.5-A.7, per jutjar si cada candidat supervivent és realment una entitat anomenada i tipificar-la. Continua sense ser el verificador round-trip: no confirma que Whisper falli en elles, així que el resultat (`entidades_fuente_b_validadas.json`) tampoc es fusiona a la llista final.


In [12]:
sys.modules.pop("evaluate", None)
sys.path.insert(0, str(EVAL_DIR))
from evaluate import STOP

In [13]:
import spacy

# 'md' en vez de 'sm': medido sobre el corpus actual, arregla exactamente el tipo de
# fallo que el filtro automático no puede pillar -- un span que se come una palabra
# de mas ('Cucurella y Baena' -> 'Cucurella'/'Baena' separados, 'João Neves y Vitinha'
# -> 'João Neves' limpio). Los 366->355 spans totales bajan poco, pero la calidad de
# los que quedan es mejor: menos falsos positivos gramaticales (Así/Cómo/Después/
# Detrás/Existe, que 'sm' capitalizaba por ir a principio de frase, 'md' ya no los
# marca como entidad). El ruido nuevo que sí aparece (frases institucionales como
# 'Constitución de 1978', spans que cruzan frase) es del mismo tipo que ya filtran
# `motivo_descarte_b` y la validación LLM (tipo=NO_ENTIDAD), no un tipo de error nuevo.
nlp = spacy.load("es_core_news_md")
LABELS_INTERES = {"PER", "ORG", "LOC"}
MAX_TOKENS_ENT = 4
# POS que delatan que el span no es una entidad real, aunque vaya capitalizado por
# posición: verbo/pronombre/adverbio colado ("Existe", "Detrás", "qué deberían") o
# span sobre-extendido sobre lo que sigue ("Athletic Club superó").
POS_NO_ENTIDAD = {"VERB", "AUX", "PRON", "ADV", "SCONJ", "CCONJ", "INTJ"}

candidatos_b = defaultdict(lambda: {"fuentes": set(), "freq": 0, "docs": set(), "labels": Counter()})


def ficheros_corpus(root_dir: Path):
    """Todos los ficheros del corpus, con o sin extensión. Un `rglob("*.txt")` se
    saltaba en silencio los documentos guardados sin extensión (p.ej.
    `RTVE/analisis-temperaturas-espana`): 1 de cada 4 ficheros no llegaba al NER."""
    return sorted(p for p in root_dir.rglob("*") if p.is_file() and not p.name.startswith("."))


def clave_entidad(ent) -> str:
    """Forma superficial sin puntuación de borde. spaCy a veces incluye el punto
    final en el span y sin esto 'Gobierno.' y 'Gobierno' entran como dos candidatos
    distintos (y se validan dos veces con el LLM).

    El punto final SÍ se conserva en las abreviaturas ('EE.UU.'), que se reconocen
    porque ya llevan otro punto dentro: quitárselo rompe la clave con la que las
    identifican `entidades_verificades_roundtrip.json` y el resto del pipeline."""
    texto = ent.text.strip()
    sin_final = texto.rstrip(',.;:¿?¡!()"\'«»…—- ')
    if texto.endswith(".") and "." in sin_final:
        return sin_final + "."          # EE.UU. / S.A. / EE.UU..  -> EE.UU.
    return sin_final.lstrip(',.;:¿?¡!()"\'«»…—- ')


def extraer_entidades_corpus(root_dir: Path):
    # Guardamos el Span de spaCy, no solo el texto: el filtro necesita el POS de
    # cada token y los límites de frase, que se pierden al pasar a str.
    encontrados, textos = [], []
    for txt_path in ficheros_corpus(root_dir):
        texto = txt_path.read_text(encoding="utf-8", errors="ignore")
        textos.append(texto)
        for ent in nlp(texto).ents:
            if ent.label_ in LABELS_INTERES:
                encontrados.append((txt_path.name, ent))
    print(f"corpus: {len(textos)} ficheros, {sum(len(t.split()) for t in textos)} palabras")
    return encontrados, textos


DIR_CORPUS = ROOT / "lab/proves_inicials/proves/raw_text"
if not DIR_CORPUS.is_dir():
    raise FileNotFoundError(
        f"No encuentro el corpus en {DIR_CORPUS}. La Fuente B no puede generar "
        "candidatos sin texto crudo."
    )

entidades_ner, textos_corpus = extraer_entidades_corpus(DIR_CORPUS)
if not textos_corpus:
    raise RuntimeError(f"{DIR_CORPUS} existe pero no contiene ningun .txt legible.")

# Uso mayúscula/minúscula en el PROPIO corpus de B (mismo truco que en A.1)
may_b, min_b = Counter(), Counter()
for texto in textos_corpus:
    for frase in re.split(r"(?<=[.!?])\s+", texto):
        for i, palabra in enumerate(frase.split()):
            tok = palabra.strip(',.;:¿?¡!()"\'«»…—-')
            if not tok or i == 0:
                continue
            n = norm(tok)
            if n:
                (may_b if tok[0].isupper() else min_b)[n] += 1


def _elision_catalana(t: str) -> re.Match | None:
    """Elisión catalana/valenciana con apóstrofo delante de vocal: el artículo
    ('d\'', 'l\'', 's\'', 'n\'') no cuenta como palabra sin capitalizar, lo que
    importa es cómo empieza lo que sigue ('d\'Empúries' -> 'Empúries')."""
    return re.match(r"^[dlsn]'(.+)$", t, re.IGNORECASE)


def _capitalizada_o_conector(t: str) -> bool:
    if t[:1].isupper() or norm(t) in STOP:
        return True
    m = _elision_catalana(t)
    return bool(m and m.group(1)[:1].isupper())


def motivo_descarte_b(ent) -> str | None:
    """Devuelve el motivo por el que el span NO es una entidad válida, o None si pasa el filtro."""
    texto = clave_entidad(ent)
    palabras = texto.split()
    toks = norm(texto).split()
    if not toks or len(toks) > MAX_TOKENS_ENT:
        return "longitud"

    # 'De la Fuente' o 'los Aliados' empiezan por un stopword pero SON la entidad
    # (apellido con partícula, apodo con artículo): solo se descarta si, quitando el
    # conector inicial, no queda ninguna palabra capitalizada -- si queda alguna, el
    # núcleo del nombre está ahí y se deja pasar al resto de filtros. Los casos
    # ambiguos que cuelan ('la Ley') los resuelve después la validación LLM
    # (tipo=NO_ENTIDAD), igual que el resto del ruido gramatical.
    if toks[0] in STOP and not any(p[:1].isupper() for p in palabras[1:]):
        return "empieza_stop"

    if len(toks) == 1:
        # Siglas cortas reales ('UE', 'PP', 'G7') no son ruido de alineamiento: se
        # detectan porque van enteras en mayúscula, a diferencia de un fragmento
        # capitalizado solo por ir al principio de frase ('Así', 'Eso').
        es_sigla = palabras[0].isupper()
        if len(toks[0]) < 4 and not es_sigla:
            return "muy_corta"
        if min_b[toks[0]] > 0 and min_b[toks[0]] >= may_b[toks[0]] * 0.5:
            return "nombre_comun"             # aparece en minúscula en el propio corpus

    # El span cruza un salto de frase/párrafo: spaCy arrastra la primera palabra
    # del bloque siguiente ("Ucrania\nEfectivamente"). El truco mayús/minús de
    # arriba no lo ve porque compara palabra a palabra, no el span completo.
    if "\n" in ent.text or any(tok.is_sent_start for tok in ent[1:]):
        return "cruza_frase"

    # Cada palabra debe ir capitalizada, salvo conectores ('de', 'del', 'los'...) y
    # elisiones catalanas ('d\'Empúries'). Si alguna no lo es, es un sintagma común
    # colado ("nueva generación") o un span sobre-extendido ("Athletic Club superó")
    # -- CON UNA EXCEPCIÓN: topónimos con núcleo genérico en minúscula ('estrecho de
    # Ormuz', 'golfo de México'), patrón habitual en español a diferencia del inglés.
    a_revisar = palabras
    if (len(palabras) >= 3 and not palabras[0][:1].isupper()
            and palabras[1].lower() in ("de", "del")):
        a_revisar = palabras[1:]
    if not all(_capitalizada_o_conector(t) for t in a_revisar):
        return "no_capitalizado"

    # Verbo/pronombre/adverbio en el span: aunque vaya capitalizado por ir al
    # principio de frase, no es una entidad ("Existe", "Detrás", "qué deberían").
    if any(tok.pos_ in POS_NO_ENTIDAD for tok in ent):
        return "pos_no_nominal"

    return None


descartes_b = defaultdict(list)
for doc, ent in entidades_ner:
    motivo = motivo_descarte_b(ent)
    if motivo:
        descartes_b[motivo].append(clave_entidad(ent) or ent.text.strip())
        continue
    texto = clave_entidad(ent)
    if not texto:
        continue
    candidatos_b[texto]["fuentes"].add("B_ner_corpus")
    candidatos_b[texto]["freq"] += 1
    candidatos_b[texto]["docs"].add(doc)
    # La etiqueta se vota entre todas las apariciones: asignarla suelta hacía que
    # ganase siempre la ÚLTIMA ocurrencia, aunque fuese la minoritaria.
    candidatos_b[texto]["labels"][ent.label_] += 1

print(f"spans NER detectados : {len(entidades_ner)}")
print(f"\n{'motivo':16} {'n':>4}   ejemplos")
print("-" * 70)
for motivo, items in sorted(descartes_b.items(), key=lambda kv: -len(kv[1])):
    print(f"{motivo:16} {len(items):4}   {', '.join(items[:5])}")
print("-" * 70)
print(f"descartados por ruido: {sum(len(v) for v in descartes_b.values())}")
print(f"candidatos B únicos  : {len(candidatos_b)}")
for ent, datos in candidatos_b.items():
    datos["label"] = datos["labels"].most_common(1)[0][0]

# Formas parciales: 'Ronaldo' y 'Cristiano Ronaldo' son el mismo referente y hoy
# consumen dos validaciones LLM y dos verificaciones round-trip. No se fusionan
# (Whisper puede fallar solo en una de las dos), pero se anotan para poder agrupar
# y priorizar aguas abajo.
# Solo se fusionan PERSONAs ('Ronaldo' -> 'Cristiano Ronaldo'): ahí compartir una
# palabra SÍ implica ser el mismo referente casi siempre. Para LUGAR/ORG no vale la
# misma suposición: 'Almería' (provincia) y 'Almería Aeropuerto' comparten palabra
# pero son entidades DISTINTAS, igual que 'España' vs 'Gobierno de España' o
# 'Austria' vs 'Ante Austria' (esto último ni siquiera es una entidad real). Fusionar
# ahí no ahorraba presupuesto de verificación, perdía cobertura de una entidad
# genuinamente distinta.
for corta, datos in candidatos_b.items():
    if len(corta.split()) != 1 or datos["label"] != "PER":
        continue
    largas = [l for l in candidatos_b
              if l != corta and corta in l.split() and candidatos_b[l]["label"] == "PER"]
    if largas:
        datos["forma_larga"] = max(largas, key=lambda l: candidatos_b[l]["freq"])

parciales = {c: d["forma_larga"] for c, d in candidatos_b.items() if d.get("forma_larga")}
print(f"\nsolapan con la Fuente A: {sorted(set(candidatos_b) & set(candidatos))}")
print(f"formas parciales detectadas ({len(parciales)}): " +
      ", ".join(f"{c} -> {l}" for c, l in list(parciales.items())[:6]))
print("\nmuestra:", sorted(candidatos_b)[:25])

/home/ugiat/.virtualenvs/sintetic/lib/python3.10/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'es_core_news_md' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


corpus: 4 ficheros, 5648 palabras
spans NER detectados : 355

motivo              n   ejemplos
----------------------------------------------------------------------
nombre_comun        8   Récords, Roja, Estamos, Estado, Además
cruza_frase         5   Ucrania
Efectivamente, Gobierno, Gobierno, Gobierno, Organización
Los miembros del
pos_no_nominal      3   O Rosal, Pasamos, Sumar
no_capitalizado     3   presidente Trump, Gobierno central, Constitución de 1978
longitud            2   Centre Delàs de Estudios para la Paz, Comisión General de Secretarios de Estado y Subsecretarios
----------------------------------------------------------------------
descartados por ruido: 21
candidatos B únicos  : 167

solapan con la Fuente A: ['Lamine Yamal', 'Pau Cubarsí']
formas parciales detectadas (6): Simón -> Unai Simón, Cucurella -> Marc Cucurella, Ronaldo -> Cristiano Ronaldo, Oyarzabal -> Mikel Oyarzabal, Rutte -> Mark Rutte, Tovar -> Juan Tovar

muestra: ['AEMET', 'Abadiño', 'Administración G

### B.4 — Validació de grafia i utilitat amb LLM

Mateix patró que A.5-A.7, però adaptat: aquí no hi ha "ground truth vs Whisper", només spans que un NER petit (`es_core_news_sm`) ha marcat com PER/ORG/LOC sobre text cru. El filtre automàtic (`motivo_descarte_b`) ja treu bastant soroll lèxic i gramatical, però no pot jutjar si un span és *semànticament* una entitat real ("Athletic Club" sí, un topònim menor mal etiquetat no) ni tipificar-la.

Reutilitza el client, l'esquema i les funcions d'A.5-A.6 (`buscar_contexto`, `_llamar_lote`, `validar_grafias`, `tipo_cambio`), amb un prompt de sistema propi (`SYSTEM_PROMPT_VALIDACION_B`) que reflecteix que aquí el punt de partida és molt més sorollós.

El LLM **només jutja grafia i tipus**, no si l'entitat val la pena. El camp `util_dataset` es va retirar: decidia per fama i tirava 51 entitats reals amb confiança alta, 12 de les quals el round-trip va mesurar després que fallen (8 al 100%). Qui decideix si entra al dataset és `src/verify_entities.py`, amb àudio.

**Això no és el verificador round-trip de "Propostes a futur".** Continua sense haver-hi prova que Whisper falli en aquestes entitats — només confirma que l'span és una entitat anomenada real i ben escrita. Per això el resultat es desa a part i **tampoc es fusiona** a `entidades_candidatas.json`.


In [14]:
# B.5 — Prompt, contexto y payload de validación para Fuente B
frases_b = []
for texto in textos_corpus:
    frases_b += [f.strip() for f in re.split(r"(?<=[.!?])\s+", texto) if f.strip()]

SYSTEM_PROMPT_VALIDACION_B = """Eres un experto en entidades nombradas del ámbito informativo español (RNE).

Recibes spans detectados automáticamente por un NER (spaCy, modelo pequeño `es_core_news_sm`) sobre un corpus de noticias en bruto. A diferencia de un ground truth humano alineado contra transcripciones, aquí NO hay ninguna garantía de que el span sea una entidad real: el NER pequeño genera bastante ruido (fragmentos que arrastran la palabra siguiente, sintagmas comunes, verbos/pronombres/adverbios colados en el span, palabras sueltas mal etiquetadas por ir en mayúscula al principio de frase). `label_ner` es la etiqueta que le puso spaCy (PER/ORG/LOC), pero puede estar equivocada o no aplicar en absoluto.

Para cada entrada devuelve:
- grafia_correcta: la grafía canónica y correctamente acentuada de la entidad. Usa el contexto para desambiguar. Si el span no es una entidad real, devuelve la entrada tal cual.
- gt_erroneo: true si la grafía correcta difiere de la entrada recibida.
- tipo: PERSONA, LUGAR, ORGANIZACION, EVENTO, OTRO, o NO_ENTIDAD si el span no es un nombre propio (fragmento de frase, sintagma común, verbo/pronombre suelto, span mal cortado, etc.).
- entidad_conocida: true si reconoces la entidad y confías en la `grafia_correcta` que devuelves; false si no la reconoces con seguridad (en ese caso sigue la REGLA CRÍTICA de abajo).
- motivo: una frase breve justificando la decisión.

REGLA CRÍTICA: si no reconoces la entidad con seguridad, NO inventes una grafía. Devuelve grafia_correcta igual a la entrada recibida, gt_erroneo=false y entidad_conocida=false."""

b_validar = [
    {
        "entrada": ent,
        "label_ner": datos["label"],
        "veces": datos["freq"],
        "contexto": buscar_contexto(ent, frases_b),   # frases_b: corpus de B, no el GT de A
    }
    for ent, datos in candidatos_b.items()
]
print(f"{len(b_validar)} entidades de Fuente B a validar con {MODEL_VALIDACION}")
print(json.dumps(b_validar[:3], indent=2, ensure_ascii=False))

167 entidades de Fuente B a validar con gpt-4o-mini
[
  {
    "entrada": "España",
    "label_ner": "LOC",
    "veces": 33,
    "contexto": "Récords y análisis de las temperaturas en España\n\nLa temperatura máxima en España hoy es de 38,1°C (8,2°C por encima de lo normal) y la mínima de 20,4°C (3,7°C por encima de lo normal)."
  },
  {
    "entrada": "Puerto de Leitariegos",
    "label_ner": "LOC",
    "veces": 1,
    "contexto": "Las estaciones donde el calor del día es más inusual para la fecha son Puerto de Leitariegos (Asturias), con 39,0°C (22,8° sobre su mediana y la más alta en sus 20 años de serie para este día); Abadiño, Urkiola (Bizkaia)…"
  },
  {
    "entrada": "Asturias",
    "label_ner": "LOC",
    "veces": 1,
    "contexto": "Las estaciones donde el calor del día es más inusual para la fecha son Puerto de Leitariegos (Asturias), con 39,0°C (22,8° sobre su mediana y la más alta en sus 20 años de serie para este día); Abadiño, Urkiola (Bizkaia)…"
  }
]


In [15]:
# B.6 — Ejecutar la validación por lotes (esta celda GASTA API)
validaciones_b = validar_grafias(b_validar, system_prompt=SYSTEM_PROMPT_VALIDACION_B)

  lote 1: 25 entidades  (coste acumulado $0.0012)
  lote 2: 25 entidades  (coste acumulado $0.0024)
  lote 3: 25 entidades  (coste acumulado $0.0034)
  lote 4: 25 entidades  (coste acumulado $0.0045)
  lote 5: 25 entidades  (coste acumulado $0.0056)
  lote 6: 25 entidades  (coste acumulado $0.0066)
  lote 7: 17 entidades  (coste acumulado $0.0074)

Total: 167 entidades en 83.7s  |  coste $0.0074


In [16]:
# B.7 — Aplicar la validación y guardar
for v in validaciones_b:
    v["tipo_cambio"] = tipo_cambio(v["entrada"], v["grafia_correcta"])
    v["forma_larga"] = candidatos_b.get(v["entrada"], {}).get("forma_larga")

validadas_b = [v for v in validaciones_b if v["tipo"] != "NO_ENTIDAD" and v["entidad_conocida"]]
revisar_b = [v for v in validaciones_b if v["tipo"] != "NO_ENTIDAD" and not v["entidad_conocida"]]
rechazadas_b = [v for v in validaciones_b if v["tipo"] == "NO_ENTIDAD"]

print(f"listas para el dataset : {len(validadas_b)}")
print(f"revisión humana        : {len(revisar_b)}")
print(f"rechazadas por el LLM  : {len(rechazadas_b)}")

print("\n--- Reparto por tipo (validadas) ---")
for tipo, n in Counter(v["tipo"] for v in validadas_b).most_common():
    print(f"  {tipo:15} {n}")

print("\n--- Rechazadas por el LLM (muestra) ---")
for v in rechazadas_b[:15]:
    print(f"  {v['entrada']:30} [{v['tipo']}] {v['motivo'][:65]}")

path_val_b = ROOT / "lab/entitats/entidades_fuente_b_validadas.json"
path_val_b.write_text(
    json_compacto({"validadas": validadas_b, "revisar_humano": revisar_b, "rechazadas": rechazadas_b}),
    encoding="utf-8",
)
print(f"\nGuardado en {path_val_b}")
print("(sigue sin fusionarse en entidades_candidatas.json: falta el verificador round-trip)")

listas para el dataset : 152
revisión humana        : 10
rechazadas por el LLM  : 5

--- Reparto por tipo (validadas) ---
  LUGAR           65
  PERSONA         53
  ORGANIZACION    30
  OTRO            3
  EVENTO          1

--- Rechazadas por el LLM (muestra) ---
  Mundiales                      [NO_ENTIDAD] No es un nombre propio, sino un término genérico.
  Simón                          [NO_ENTIDAD] No es un nombre completo, sino un apellido.
  Baena                          [NO_ENTIDAD] No es un nombre completo, sino un apellido.
  Frente a Austria               [NO_ENTIDAD] No es un nombre propio, sino una frase que indica una situación.
  secretario de Defensa          [NO_ENTIDAD] No es un nombre propio.

Guardado en /media/ugiat/dd2/projects/nerea/sintetic_dataset/Sintetic-dataset/lab/entitats/entidades_fuente_b_validadas.json
(sigue sin fusionarse en entidades_candidatas.json: falta el verificador round-trip)


## Font C — Fragmentació del tokenizer de Whisper

El BPE de Whisper trenca les paraules poc freqüents en diversos subtokens. Com més subtokens per caràcter, més probable que el model no hagi vist la paraula sencera durant l'entrenament — és un proxy barat per **ordenar** els candidats de la Font B abans de gastar TTS + inferència real.

**Compte amb el tokenizer.** Whisper no fa servir el de GPT-2 tal qual: va refer el BPE per a multilingüe, mateixa mida de vocabulari però *merges diferents*. Comparat amb `tiktoken.get_encoding("gpt2")`, els ids difereixen a totes les paraules provades i el recompte difereix justament en els noms estrangers (`Dončić` 3 vs 4, `Ancelotti` 3 vs 4), que són precisament els que aquest score pretén caçar. Per això aquí s'utilitza el tokenizer real via `transformers`.

Continua sent un prior feble, i amb un biaix concret: en normalitzar pel nombre de caràcters (`tokens / len`), **premia les paraules curtes**. A la pràctica les sigles pugen amunt (`CIDOB` 0.80, `PSOE` 0.75) per sobre de `Krasznahorkai` (0.46), que és justament el tipus de nom que hauria d'encapçalar el rànquing. Fes-lo servir com a desempat, no com a criteri principal, i no substitueix la verificació real.


In [17]:
from transformers import WhisperTokenizer

# El tokenizer real del modelo que estamos evaluando, no una aproximación con gpt2.
tok_whisper = WhisperTokenizer.from_pretrained("openai/whisper-large-v3")


def score_fragmentacion(palabra: str) -> float:
    return len(tok_whisper.encode(palabra, add_special_tokens=False)) / max(len(palabra), 1)


for ent in list(candidatos_b.keys()):
    candidatos_b[ent]["score_fragmentacion"] = round(score_fragmentacion(ent), 3)

ranking_b = sorted(candidatos_b.items(), key=lambda kv: -kv[1]["score_fragmentacion"])
print("Candidatos de B ordenados por fragmentación (los más 'raros' primero):\n")
for ent, d in ranking_b[:20]:
    print(f"  {d['score_fragmentacion']:.3f}  {ent:38} [{d['label']}] x{d['freq']}")

salida_b = {
    ent: {
        "fuentes": sorted(d["fuentes"]),
        "freq": d["freq"],
        "label": d["label"],
        "docs": sorted(d["docs"]),
        "score_fragmentacion": d["score_fragmentacion"],
        "forma_larga": d.get("forma_larga"),
    }
    for ent, d in ranking_b
}
path_b = ROOT / "lab/entitats/entidades_fuente_b_ranked.json"
path_b.write_text(json_compacto(salida_b), encoding="utf-8")
print(f"\n{len(salida_b)} candidatos de B guardados en {path_b}")
print("(pendientes de verificación: NO entran en la lista final)")

Candidatos de B ordenados por fragmentación (los más 'raros' primero):

  1.000  G7                                     [ORG] x1
  0.800  CIDOB                                  [ORG] x3
  0.750  Pelé                                   [PER] x1
  0.750  Irán                                   [LOC] x1
  0.750  RTVE                                   [ORG] x1
  0.750  PSOE                                   [ORG] x1
  0.667  Mbappé                                 [PER] x1
  0.667  EE.UU.                                 [LOC] x6
  0.600  Araba                                  [LOC] x1
  0.600  Álava                                  [LOC] x1
  0.600  AEMET                                  [ORG] x1
  0.600  Cádiz                                  [LOC] x3
  0.600  Pedri                                  [PER] x2
  0.600  Rusia                                  [LOC] x1
  0.600  Rodri                                  [PER] x1
  0.600  Rutte                                  [PER] x2
  0.600  Japón  

## Fusió i priorització final

Exporta `lab/entitats/entidades_candidatas.json`, la llista que consumeix la resta del pipeline.

| Font | Criteri d'entrada | Prioritat |
|---|---|---|
| **A** | fallades observades sobre àudio real de RNE | 10.0 + extres |
| **B** | `tasa_error >= 0.3` **mesurada** pel round-trip | 1.0 + `tasa_error` |

El verificador (`src/verify_entities.py`) li dona l'evidència a la Font B. Si encara no s'ha executat, la cel·la ho avisa i exporta només la Font A.

> **Per què no s'usa `score_fragmentacion` com a llindar.** Es va provar (`>= 0.6`) i es
> va contrastar amb les mesures reals: de 12 seleccionades només 4 fallaven de veritat, i
> deixava fora **les nou** amb `tasa_error=1.0` (`Lamine Yamal` 0.33, `Nuno Mendes` 0.36,
> `Pete Hegseth` 0.42, `OSCE` 0.50). És el biaix que la pròpia Font C documenta: en
> dividir pel nombre de caràcters premia les sigles i penalitza els noms llargs
> estrangers, que són justament l'objectiu. Serveix per desempatar, no per seleccionar.


In [ ]:
def prioridad(datos: dict) -> float:
    # La Fuente A pesa mucho más porque son fallos reales observados sobre audio de
    # RNE, no candidatos teóricos.
    peso_verificado = 10.0 if "A_errores_modelo" in datos["fuentes"] else 0.0
    peso_ner = 1.0 if "B_ner_corpus" in datos["fuentes"] else 0.0
    return peso_verificado + peso_ner + datos.get("score_fragmentacion", 0)


# 1. Fuente A: entra entera, es la que tiene evidencia real de fallo.
salida = {}
for ent, datos in candidatos.items():
    salida[ent] = {
        "fuentes": sorted(datos["fuentes"]),
        "freq": datos["freq"],
        "score_fragmentacion": datos.get("score_fragmentacion", 0),
        "prioridad": round(prioridad(datos), 3),
    }
    if "A_errores_modelo" in datos["fuentes"]:
        salida[ent]["veces_error"] = datos.get("veces_error")
        salida[ent]["variantes_erroneas"] = datos.get("variantes_erroneas")

# 2. Fuente B: solo las que el verificador round-trip ha MEDIDO que fallan.
UMBRAL_TASA_ERROR = 0.3   # ver la distribución en `entidades_verificades_roundtrip.json`
PATH_ROUNDTRIP = ROOT / "lab/entitats/entidades_verificades_roundtrip.json"

if PATH_ROUNDTRIP.exists():
    verificades = json.loads(PATH_ROUNDTRIP.read_text(encoding="utf-8"))["resultats"]
    medidas = {r["entitat"]: r["tasa_error"] for r in verificades if r.get("tasa_error") is not None}

    n_inyectadas = 0
    for ent, tasa in medidas.items():
        if tasa < UMBRAL_TASA_ERROR or ent in salida:
            continue
        # Prioridad por debajo de la Fuente A (10.0) pero ordenada por error medido.
        salida[ent] = {
            "fuentes": ["B_ner_corpus_verificada"],
            "freq": candidatos_b.get(ent, {}).get("freq", 0),
            "score_fragmentacion": candidatos_b.get(ent, {}).get("score_fragmentacion", 0),
            "tasa_error": tasa,
            "prioridad": round(1.0 + tasa, 3),
        }
        n_inyectadas += 1
    print(f"Fuente B: {n_inyectadas} entidades inyectadas "
          f"(tasa_error >= {UMBRAL_TASA_ERROR}, de {len(medidas)} verificadas)")
else:
    print(f"Sin round-trip en {PATH_ROUNDTRIP.name}: solo se exporta la Fuente A.\n"
          f"Genéralo con: python3 src/verify_entities.py")

salida_ordenada = dict(sorted(salida.items(), key=lambda kv: kv[1]["prioridad"], reverse=True))

# Ruta canónica: es la que leen `dictionary.ipynb` y `src/verify_entities.py`.
output_path = ROOT / "lab/entitats/entidades_candidatas.json"
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json_compacto(salida_ordenada), encoding="utf-8")

print(f"Guardadas {len(salida_ordenada)} entidades candidatas en {output_path}")
list(salida_ordenada.items())[:10]

Fuente B: 41 entidades inyectadas (tasa_error >= 0.3, de 147 verificadas)
Guardadas 268 entidades candidatas en /media/ugiat/dd2/projects/nerea/sintetic_dataset/Sintetic-dataset/lab/entitats/entidades_candidatas.json


[('Lamine Yamal',
  {'fuentes': ['A_errores_modelo'],
   'freq': 41,
   'score_fragmentacion': 0,
   'prioridad': 10.0,
   'veces_error': 41,
   'variantes_erroneas': ['alhamid',
    'alhamid jamal',
    'cono sin la miña mal',
    'jamal',
    'la',
    'la mil',
    'la mim',
    'la minya',
    'la minya mal',
    'la minyamal',
    'la miña',
    'la miña mal',
    'lamin',
    'lamín',
    'llamal',
    'meñamal',
    'minyamal',
    'miña']}),
 ('Pérez Llorca',
  {'fuentes': ['A_errores_modelo'],
   'freq': 24,
   'score_fragmentacion': 0,
   'prioridad': 10.0,
   'veces_error': 24,
   'variantes_erroneas': ['de yorca',
    'peret',
    'pérez llota mazón de verdad',
    'yerka',
    'yorca',
    'york']}),
 ('Núñez Feijóo',
  {'fuentes': ['A_errores_modelo'],
   'freq': 22,
   'score_fragmentacion': 0,
   'prioridad': 10.0,
   'veces_error': 22,
   'variantes_erroneas': ['contrafijo',
    'feijo',
    'feijo dijo',
    'feijó',
    'fejo',
    'fejó',
    'fijo',
    'fijón',
  